In [ ]:
import translators as ts
import time
from nltk.translate import bleu_score
import numpy as np
from bleurt import score
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pathlib

In [ ]:
import datasets

Get SMOLSent En->Cantonese data

In [ ]:
rel_path = pathlib.Path(".")
en_yue_sentences_validation = pd.read_json(path_or_buf=rel_path / "smoldata/smol_en_yue_sentences/validation.json", lines=True)

In [ ]:
yue_zh_characters_validation = datasets.load_dataset("google/smol", "gatitos__yue_zh")

Benchmark sentences on various translation sites using bleu to demonstrate that chinese models don't necessarily outperform cantonese models

In [ ]:
# run this to skip scraping
with open(r"translation_service_en_yue_sentence_validation.json", "r") as f:
    all_trans = json.load(f)

In [ ]:
services = ["bing", "google"]

langs = {
    "bing": ["chinese (simplified)", "chinese (traditional)", "cantonese"],
    "google": ["chinese"]
}

lang_map = {
    "bing": {"chinese (simplified)" : "zh-Hans", "chinese (traditional)": "zh-Hant", "cantonese": "yue", "english": "en"},
    "google": {"chinese": "zh", "english": "en"}
}

all_trans = []

for index, sentence in tqdm(en_yue_sentences_validation.iterrows(), total=en_yue_sentences_validation.shape[0]):
    
    translations = {
        "original_eng": sentence["src"],
        "original_yue": sentence["trg"]
    }

    for service in services:

        attempts = 0
        while attempts < 3:
            try:
                trans = {language: {"translation":ts.translate_text(sentence["trg"], translator="bing", from_language=lang_map[service][language], to_language=lang_map[service]["english"])} for language in langs[service] }
                translations[service] = trans
                break
            except KeyboardInterrupt:
                sys.exit(1)
            except:
                tqdm.write(f"error on sentece {sentence['id']}, retrying... ({attempts + 1})")
                attempts += 1
                time.sleep(3)
        if attempts == 3:
            tqdm.write(f"skipped sentence {sentence['id']} after {attempts+1} attempts")
            continue
        
    all_trans.append(translations)

In [ ]:
with open(r"translation_service_en_yue_sentence_validation.json", "w") as f:
    json.dump(all_trans, f, indent=4)

In [ ]:
def bleu(ref, cand):
    return bleu_score.sentence_bleu(
            [ref.split()], 
            cand.split(), 
                #use smoothing method 7 that had best chinese->english human-evaluation corerlation from https://aclanthology.org/W14-3346/
            smoothing_function= bleu_score.SmoothingFunction().method7 
            )

def benchmark(all_trans, metric, metric_name): 
    with tqdm(total = len(all_trans) * np.sum([len(langs[service]) for service in services]), leave=True) as pbar:
        for i, trans in enumerate(all_trans):
            for service in services:
                for lang in langs[service]:
                    all_trans[i][service][lang][metric_name] = metric(trans["original_eng"], trans[service][lang]["translation"])
                    pbar.update()

def print_benchmark(all_trans, metric_name):
    for service in services:
            for lang in langs[service]:
                print(f'{service:10s} - {lang:25s}: {np.mean([trans[service][lang][metric_name] for trans in all_trans]):.4f} {metric_name}')

In [ ]:
benchmark(all_trans, bleu, "bleu")
print_benchmark(all_trans, "bleu")

Since BLEU is a n-gram based evaluation proxy it evaluates sentences badly when they don't share vocabulary or word order with the reference, even if the semantic meaning is conserved. This can be seen below where Bing's chinese traditional model scores vastly above the others even though it has the same semantic meaning.  

In [ ]:
def print_sentences(sent_trans):
    print(f'{"original":29s} : {sent_trans["original_eng"]}')
    [print(f'bing   {lang:22s} : {sent_trans["bing"][lang]["translation"]}') for lang in langs["bing"]]
    [print(f'google {lang:22s} : {sent_trans["google"][lang]["translation"]}') for lang in langs["google"]]
    print(f'{"original yue":29s} : {sent_trans["original_yue"]}')

In [ ]:
i = 8
print_sentences(all_trans[i])
print("\n")
print_benchmark([all_trans[i]], "bleu")

For that reason we instead switch to the model-based evaluation metric BLEURT

In [ ]:
scorer = score.BleurtScorer("bleurt-base-128")

In [ ]:
def bleurt(ref, cand):
    return scorer.score(references=[ref], candidates=[cand])[0]

benchmark(all_trans, bleurt, "bleurt")
print_benchmark(all_trans, "bleurt")

In [ ]:
i = 8
print_sentences(all_trans[i])
print("\n")
print_benchmark([all_trans[i]], "bleurt")

Now we can see that we have a better evaluation measure, the two translations that had the same semantic meaning but scored much worse now have around the same score, and the translation that did not translate "rating" scored worse than the others. 

Next we will use the bleurt evaluation metric to filter for bad translations that we can look for patterns in. 

In [ ]:
filter_value = -0.3
bad_translations = [trans for trans in all_trans if np.all([np.all([trans[service][lang]["bleurt"] < filter_value for lang in langs[service]]) for service in services])]
print_sentences(bad_translations[0])

Many bad translations mistranslate words, so we believe it is likely the case that bad translations have more cantonese-specific characters in them, so we plot this relation to see if there is any correlation. 

In [ ]:
with open("character_classification.json", "r") as f:
    character_classfication = json.load(f)

In [ ]:
def canto_characters_in_sentence(sentence):
    canto_only = 0
    for x in sentence:
        if x in character_classfication.keys():
            if character_classfication[x] == "CANTO_ONLY":
                canto_only += 1
    return canto_only, len(sentence)

all_translations_canto_score = {}
for service in services:
    
    all_translations_canto_score[service] = {}
    for lang in langs[service]:
        all_translations_canto_score[service][lang] = []
        for i in range(len(all_trans)):
            score = all_trans[i][service][lang]["bleurt"]
            canto,length = canto_characters_in_sentence(all_trans[i]["original_yue"])
            all_translations_canto_score[service][lang].append((canto/length, score))

In [ ]:
fig, axs = plt.subplots(2,2,sharey=True,sharex=True)

fig_map = {"bing": {"chinese (simplified)": (0,0), "chinese (traditional)": (1,0), "cantonese": (0,1)}, "google": {"chinese": (1,1)}}

hexbin_size = 10
for service in services:
    for lang in langs[service]:
        z = axs[fig_map[service][lang]].hexbin(
    np.array(all_translations_canto_score[service][lang])[:,0].flatten(), 
    np.array(all_translations_canto_score[service][lang])[:,1].flatten(), 
    gridsize=hexbin_size,
    #cmap="ciridis", 
    mincnt=1
    )

        axs[fig_map[service][lang]].set_title(service + "_" + lang)
        axs[fig_map[service][lang]].margins(0)

cb = fig.colorbar(z, ax=axs, label='counts')


fig.supxlabel("percentage cantonese characters in sentence")
fig.supylabel("bleurt")
fig.suptitle("bleurt vs cantonese character presence")

In [ ]:
bins = [0]

v = np.array(all_translations_canto_score["bing"]["chinese (simplified)"] + all_translations_canto_score["bing"]["chinese (traditional)"] + all_translations_canto_score["bing"]["cantonese"] + all_translations_canto_score["google"]["chinese"])

bin_ranges = [0, 0.0001, 0.05, 0.1, 0.15, 0.2, 1]


bins = [np.argwhere((v[:,0] >= bin_ranges[i]) & (v[:,0] < bin_ranges[i+1]) ) for i in range(len(bin_ranges) - 1)]

ax = sns.violinplot([v[bin, 1].flatten() for bin in bins])
xtick_labels = [str(bin_ranges[i]) + "-" + str(bin_ranges[i+1]) for i in range(len(bin_ranges)-1)]
xtick_labels[0] = "0"
xtick_labels[1] = "0-0.05"
ax.set_xticklabels(xtick_labels)

ax.set_ylabel("bleurt")
ax.set_xlabel("bins")
ax.set_title("bing & google model translations")

Generally the models do very well when there are no cantonese characters. Beyond that there may be a slight negative correlation between average performance and the number of cantonese characters. 

We do another test with a local large language model (MADLAD400 3B-mt)

In [ ]:
with open(r"sentence_translations_madlad400-3b.json", "r") as f:
    madlad_translations = json.load(f)

In [ ]:
for x in tqdm(madlad_translations):
    x["bleurt"] = bleurt(x["original"], x["cantonese"])
    canto, length = canto_characters_in_sentence(x["original_yue"])
    x["canto_percent"] = canto/length

In [ ]:
[np.mean([x["bleurt"] for x in madlad_translations])]

In [ ]:
madlad_relation = np.array([(x["canto_percent"], x["bleurt"]) for x in madlad_translations])
z = plt.hexbin(madlad_relation[:,0], madlad_relation[:,1], gridsize=hexbin_size, mincnt=1)

plt.colorbar(z, label="counts")
plt.margins(0)

plt.xlabel("percentage cantonese characters in sentence")
plt.ylabel("bleurt")

In [ ]:
v = np.array(madlad_relation)

bin_ranges = [0, 0.001, 0.05, 0.1, 0.15, 0.2, 1]

bins = [np.argwhere((v[:,0] >= bin_ranges[i]) & (v[:,0] < bin_ranges[i+1]) ) for i in range(len(bin_ranges) - 1)]

ax = sns.violinplot([v[bin, 1].flatten() for bin in bins])
xtick_labels = [str(bin_ranges[i]) + "-" + str(bin_ranges[i+1]) for i in range(len(bin_ranges)-1)]
xtick_labels[0] = "0"
xtick_labels[1] = "0-0.5"
ax.set_xticklabels(xtick_labels)

ax.set_ylabel("bleurt")
ax.set_xlabel("bins")
ax.set_title("madlad400 3B-mt translations")

We see the same pattern, the model does well on sentences that only contain mandarin characters, somewhat well on sentences that contain a small amount of cantonese characters and all over the place on other sentences. The negative correlation between number of cantonese characters and bleurt score is more apparent on this model. 

In [ ]:
def canto_characters_in_sentence(sentence):
    canto_chars = []
    for x in sentence:
        if x in character_classfication.keys():
            if character_classfication[x] == "CANTO_ONLY":
                canto_chars.append(x)
    return canto_chars

In [ ]:
madlad_translations[0]

In [ ]:
"長度" in yue_zh_characters_validation["train"]["src"]

In [ ]:
[x in yue_zh_characters_validation["train"]["src"] for x in canto_characters_in_sentence(madlad_translations[15]["original_yue"])]

In [ ]:
bad_translations = []
for trans in madlad_translations:
    if trans["bleurt"] < -0.5 :
        if np.all([x in yue_zh_characters_validation["train"]["src"] for x in canto_characters_in_sentence(trans["original_yue"])]):
            bad_translations.append(trans)



#bad_translations = [trans for trans in madlad_translations if trans["bleurt"] < -0.5 and np.all([x in yue_zh_characters_validation["train"]["src"] for (i, x) in canto_characters_in_sentence(trans["original_yue"])])]
len(bad_translations)

In [ ]:
[x for x in madlad_translations if x["original"] == "Rohan was my sibling who always flew a kite at noon, even if it was cloudy."]

In [ ]:
bad_translations[0]

In [ ]:
canto_characters_in_sentence(bad_translations[0]["original_yue"])

In [ ]:
[x for x in yue_zh_characters_validation["train"] if x["src"] in canto_characters_in_sentence(bad_translations[0]["original_yue"]) ]